# 1st Hidden Layer — Evaluate Little-Perturbation Checkpoints (SHD & SSC)

In the 1st-layer hidden-perturbation sweeps, models trained with a *little*
perturbation tend to hold up better across the sweep than the matched
train@level / eval@level baseline. This notebook isolates that effect for the
two realistic datasets: for each dataset it loads a **single checkpoint trained
at a small perturbation level** and evaluates that one model across the
**entire** perturbation sweep (eval-on-checkpoint), rather than training a fresh
model per level.

The perturbation is the hidden-layer **partial spike relocation** used in
training (``perturb_hidden_batch``): a fraction ``f`` of each neuron's spikes
are moved to randomly chosen empty time bins, with the per-neuron spike count
preserved exactly. Each dataset is evaluated twice — once for the **no-delay**
model and once for the **delay** model — using the same checkpoint level:

- **SHD** — Spiking Heidelberg Digits; default checkpoint ``f = 0.2``.
- **SSC** — Spiking Speech Commands; default checkpoint ``f = 0.2``.

Evaluations cover the **whole / part / norm** variants of each dataset and the
full sweep ``F_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]``. Per-section results
are written to ``log_eval_on_perturbatedModel/`` with a ``1stLayer`` tag so they
stay distinct from the 2nd-layer eval-on-checkpoint outputs.

In [1]:
# Setup: reuse the training modules (model classes, data pipeline and
# evaluation routines) so the evaluation matches the original sweeps.
import sys
import json
from pathlib import Path

import numpy as np
import torch

BASE_DIR = Path.cwd()
assert (BASE_DIR / "shd").is_dir() and (BASE_DIR / "ssc").is_dir(), (
    "Run this notebook from my_project/code/realistic/ so the "
    "shd / ssc packages are importable."
)

for sub in ("shd", "ssc"):
    sub_path = str((BASE_DIR / sub).resolve())
    if sub_path not in sys.path:
        sys.path.append(sub_path)

import shd_train as shd_mod
import ssc_train as ssc_mod

device = shd_mod.device
print(f"Device: {device}")

# Dataset variants to evaluate (whole / part / norm), matching the sweeps.
EVAL_DATASETS = ("whole", "part", "norm")

# Destination for the eval-on-checkpoint sweep results.
EVAL_LOG_DIR = Path("log_eval_on_perturbatedModel")
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Eval results dir: {EVAL_LOG_DIR.resolve()}")

Using device: cuda
Using device: cuda
Device: cuda
Eval results dir: D:\IC_2025\IRP\workspace\my_project\code\realistic\log_eval_on_perturbatedModel


## Shared evaluation helpers

``test_with_repeats`` takes the perturbation level ``f`` as its third positional
argument, so one evaluation routine serves both datasets and both delay modes.
Unlike the ``perturbation/`` notebooks — where three perturbation *types* shared
a single SHD data pipeline — the two datasets here have **different** loaders
(``load_shd_data`` reads ``.mat``, ``load_ssc_data`` reads ``.h5``), so the
helpers are parameterised by a *family* (``"shd"`` / ``"ssc"``) that bundles the
module, network class, loader and dataset-config key.

The 1st-layer checkpoint filename carries no ``_2ndLayer`` infix
(``{family}_{dataset}_{delay}_{token}.pt``), while the output JSON is tagged
with ``1stLayer`` so both layers can share ``EVAL_LOG_DIR``.

In [2]:
# Per-dataset evaluation specs. Each family bundles everything that differs
# between SHD and SSC: the training module, the network class, the loader
# function and the DATASET_CONFIGS key holding the data-file path.
DATASET_FAMILIES = {
    "shd": {
        "module": shd_mod,
        "net_class": shd_mod.SHDNetwork,
        "load_fn": shd_mod.load_shd_data,
        "file_key": "mat_file",
    },
    "ssc": {
        "module": ssc_mod,
        "net_class": ssc_mod.SSCNetwork,
        "load_fn": ssc_mod.load_ssc_data,
        "file_key": "h5_file",
    },
}

_TEST_LOADER_CACHE: dict[tuple[str, str], object] = {}


def get_test_loader(family: str, dataset_key: str):
    """Return the test DataLoader for ``(family, dataset_key)``.

    Uses the same split ranges and seed as training, so the evaluation set
    matches the one behind the original sweep results. Cached because the
    test data is identical across delay modes and checkpoint levels.
    """
    cache_key = (family, dataset_key)
    if cache_key not in _TEST_LOADER_CACHE:
        spec = DATASET_FAMILIES[family]
        module = spec["module"]
        cfg = module.DATASET_CONFIGS[dataset_key]
        X, Y = spec["load_fn"](
            cfg[spec["file_key"]], target_T=module.SIM_PARAMS["tSample"]
        )
        _, _, test_loader = module.build_dataloaders(
            X, Y, batch_size=module.BATCH_SIZE, seed=module.SEED
        )
        _TEST_LOADER_CACHE[cache_key] = test_loader
    return _TEST_LOADER_CACHE[cache_key]


def load_checkpoint(module, net_class, dataset_key, ckpt_filename, use_delay):
    """Instantiate a network (delay or no-delay) and load a saved checkpoint."""
    cfg = module.DATASET_CONFIGS[dataset_key]
    net = net_class(
        input_dim=cfg["input_dim"],
        hidden_units=module.HIDDEN_UNITS,
        num_classes=module.NUM_CLASSES,
        use_delay=use_delay,
        max_delay=module.MAX_DELAY,
    ).to(module.device)
    ckpt_path = module.DATA_DIR / ckpt_filename
    state = torch.load(ckpt_path, map_location=module.device)
    net.load_state_dict(state)
    net.eval()
    return net


def evaluate_across_levels(family, module, net, dataset_key, eval_levels):
    """Evaluate one fixed model at every perturbation level."""
    test_loader = get_test_loader(family, dataset_key)
    results = {}
    for level in eval_levels:
        res = module.test_with_repeats(net, test_loader, level)
        results[level] = res
        print(f"      eval@{level}: {res['mean']:.4f} +/- {res['std']:.4f}")
    return results


def save_sweep_json(results, out_path, key_fn):
    """Serialise eval results to the same schema as the training sweeps."""
    serial = {
        key_fn(level): {
            "mean": float(d["mean"]),
            "std": float(d["std"]),
            "values": [float(v) for v in d["values"]],
        }
        for level, d in results.items()
    }
    with open(out_path, "w") as fp:
        json.dump(serial, fp, indent=2)
    print(f"  saved -> {out_path}")


def run_eval(family, delay_tag, ckpt_token, key_fn):
    """Evaluate the ``family``/``delay_tag`` checkpoint across all levels.

    Loads ``{family}_{ds}_{delay_tag}_{ckpt_token}.pt`` for
    every dataset, evaluates it at each level in ``module.F_VALUES`` and writes
    one ``{family}_1stLayer_{ds}_{delay_tag}_evalon_{ckpt_token}.json``
    per dataset to ``EVAL_LOG_DIR``.
    """
    spec = DATASET_FAMILIES[family]
    module = spec["module"]
    net_class = spec["net_class"]
    use_delay = delay_tag == "delay"
    for dataset_key in EVAL_DATASETS:
        ckpt_file = (
            f"{family}_{dataset_key}_{delay_tag}_{ckpt_token}.pt"
        )
        print(f"[{family}/{delay_tag}] dataset={dataset_key} "
              f"| checkpoint={ckpt_file}")
        net = load_checkpoint(
            module, net_class, dataset_key, ckpt_file, use_delay
        )
        results = evaluate_across_levels(
            family, module, net, dataset_key, module.F_VALUES
        )
        out_path = EVAL_LOG_DIR / (
            f"{family}_1stLayer_{dataset_key}_{delay_tag}_"
            f"evalon_{ckpt_token}.json"
        )
        save_sweep_json(results, out_path, key_fn=key_fn)

## 1a. SHD — no delay

Evaluate the **no-delay** SHD checkpoint trained at ``CHECKPOINT_F_SHD`` across
the full perturbation sweep (``shd_mod.F_VALUES``).

In [5]:
# Perturbation level of the checkpoint to evaluate (model trained at this f).
# Shared by the no-delay (1a) and delay (1b) sub-sections.
CHECKPOINT_F_SHD = 0.4
_shd_token = f"f{CHECKPOINT_F_SHD}"

run_eval(
    family="shd", delay_tag="nodelay",
    ckpt_token=_shd_token,
    key_fn=lambda lvl: str(float(lvl)),
)

[shd/nodelay] dataset=whole | checkpoint=shd_whole_nodelay_f0.4.pt
      eval@0.0: 0.4536 +/- 0.0000
      eval@0.2: 0.4429 +/- 0.0045
      eval@0.4: 0.5404 +/- 0.0061
      eval@0.6: 0.5055 +/- 0.0049
      eval@0.8: 0.4429 +/- 0.0067
      eval@1.0: 0.3825 +/- 0.0067
  saved -> log_eval_on_perturbatedModel\shd_1stLayer_whole_nodelay_evalon_f0.4.json
[shd/nodelay] dataset=part | checkpoint=shd_part_nodelay_f0.4.pt
      eval@0.0: 0.2271 +/- 0.0000
      eval@0.2: 0.2580 +/- 0.0057
      eval@0.4: 0.3801 +/- 0.0110
      eval@0.6: 0.3496 +/- 0.0068
      eval@0.8: 0.2857 +/- 0.0132
      eval@1.0: 0.2405 +/- 0.0110
  saved -> log_eval_on_perturbatedModel\shd_1stLayer_part_nodelay_evalon_f0.4.json
[shd/nodelay] dataset=norm | checkpoint=shd_norm_nodelay_f0.4.pt
      eval@0.0: 0.1221 +/- 0.0000
      eval@0.2: 0.1941 +/- 0.0065
      eval@0.4: 0.2149 +/- 0.0026
      eval@0.6: 0.1705 +/- 0.0116
      eval@0.8: 0.1262 +/- 0.0015
      eval@1.0: 0.1001 +/- 0.0043
  saved -> log_eval_on_p

## 1b. SHD — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
``CHECKPOINT_F_SHD`` across the full perturbation sweep.

In [6]:
run_eval(
    family="shd", delay_tag="delay",
    ckpt_token=_shd_token,
    key_fn=lambda lvl: str(float(lvl)),
)

[shd/delay] dataset=whole | checkpoint=shd_whole_delay_f0.4.pt
      eval@0.0: 0.8203 +/- 0.0000
      eval@0.2: 0.8105 +/- 0.0058
      eval@0.4: 0.7778 +/- 0.0060
      eval@0.6: 0.6807 +/- 0.0016
      eval@0.8: 0.4952 +/- 0.0047
      eval@1.0: 0.3380 +/- 0.0044
  saved -> log_eval_on_perturbatedModel\shd_1stLayer_whole_delay_evalon_f0.4.json
[shd/delay] dataset=part | checkpoint=shd_part_delay_f0.4.pt
      eval@0.0: 0.6361 +/- 0.0000
      eval@0.2: 0.6361 +/- 0.0096
      eval@0.4: 0.6105 +/- 0.0062
      eval@0.6: 0.5022 +/- 0.0035
      eval@0.8: 0.3378 +/- 0.0153
      eval@1.0: 0.2031 +/- 0.0060
  saved -> log_eval_on_perturbatedModel\shd_1stLayer_part_delay_evalon_f0.4.json
[shd/delay] dataset=norm | checkpoint=shd_norm_delay_f0.4.pt
      eval@0.0: 0.4054 +/- 0.0000
      eval@0.2: 0.4062 +/- 0.0040
      eval@0.4: 0.3968 +/- 0.0053
      eval@0.6: 0.2955 +/- 0.0043
      eval@0.8: 0.1803 +/- 0.0040
      eval@1.0: 0.1241 +/- 0.0057
  saved -> log_eval_on_perturbatedModel\

## 2a. SSC — no delay

Evaluate the **no-delay** SSC checkpoint trained at ``CHECKPOINT_F_SSC`` across
the full perturbation sweep (``ssc_mod.F_VALUES``).

In [ ]:
# Perturbation level of the checkpoint to evaluate (model trained at this f).
# Shared by the no-delay (2a) and delay (2b) sub-sections.
CHECKPOINT_F_SSC = 0.2
_ssc_token = f"f{CHECKPOINT_F_SSC}"

run_eval(
    family="ssc", delay_tag="nodelay",
    ckpt_token=_ssc_token,
    key_fn=lambda lvl: str(float(lvl)),
)

## 2b. SSC — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
``CHECKPOINT_F_SSC`` across the full perturbation sweep.

In [ ]:
run_eval(
    family="ssc", delay_tag="delay",
    ckpt_token=_ssc_token,
    key_fn=lambda lvl: str(float(lvl)),
)